# TFM · Cuaderno 06 — descargar y ver tazas (Objaverse)

Primer contacto de principio a fin: **descargar tazas de Objaverse → cargar una → visualizarla → exportarla a STL**.

**Cómo usarlo:** en Google Colab (*Archivo → Subir cuaderno*) o en VS Code, ejecuta las celdas en orden con ▶ (o `Shift+Enter`). No necesita GPU.

_Raquel Roca · TFM Reconstrucción 3D · Junio 2026_

## 1. Instalar las librerías
`objaverse` para descargar los modelos 3D y `trimesh` para cargarlos y manejarlos.

In [ ]:
!pip install objaverse trimesh
print("Librerias instaladas")

## 2. Ver qué categorías de tazas existen
Objaverse etiqueta sus objetos con categorías (LVIS). Primero miramos cómo se llama exactamente la de tazas (puede ser `mug`, `cup`...).

In [ ]:
import objaverse

lvis = objaverse.load_lvis_annotations()   # diccionario: categoria -> lista de IDs

# Categorias relacionadas con tazas
candidatas = [k for k in lvis if 'mug' in k.lower() or 'cup' in k.lower()]
print("Categorias de tazas disponibles:", candidatas)

## 3. Elegir la categoría y ver cuántas tazas hay

In [ ]:
# Usamos 'mug' si existe; si no, la primera candidata encontrada
categoria = "mug" if "mug" in lvis else candidatas[0]
ids = lvis[categoria]
print("Categoria elegida:", categoria)
print("Tazas disponibles:", len(ids))

## 4. Descargar un subconjunto (20 tazas)
No bajamos todas: con 20 sobra para empezar y es rápido. Objaverse las descarga como archivos `.glb` (un formato 3D).

In [ ]:
subset = ids[:20]
objetos = objaverse.load_objects(uids=subset)   # devuelve {id: ruta_del_archivo}
print("Descargadas:", len(objetos), "tazas")

# Ver las rutas de las primeras
for uid, ruta in list(objetos.items())[:3]:
    print(uid, '->', ruta)

## 5. Cargar la primera taza y ver su información
Un `.glb` puede contener varias piezas; las unimos en una sola malla. Luego miramos cuántos vértices/caras tiene y si es *watertight* (cerrada).

In [ ]:
import trimesh

ruta = list(objetos.values())[0]
cargado = trimesh.load(ruta)

# Si el archivo trae varias piezas (Scene), las unimos en una malla unica
if isinstance(cargado, trimesh.Scene):
    malla = trimesh.util.concatenate([g for g in cargado.geometry.values()])
else:
    malla = cargado

print("Vertices:", len(malla.vertices))
print("Caras (triangulos):", len(malla.faces))
print("Es watertight (cerrada/imprimible)?:", malla.is_watertight)

## 6. Visualizar la taza
Colab no abre el visor 3D interactivo, así que muestreamos puntos de la superficie y los pintamos con matplotlib (una 'nube de puntos' de la taza).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

puntos = malla.sample(4000)   # 4000 puntos repartidos por la superficie

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(puntos[:, 0], puntos[:, 2], puntos[:, 1], s=1)
ax.set_title(f"Primera {categoria} (nube de puntos)")
ax.set_box_aspect([1, 1, 1])
plt.show()

## 7. Exportar a STL (el formato imprimible)
Cerramos el círculo: guardamos la taza como `.stl`, el formato que entiende una impresora 3D.

In [ ]:
malla.export("taza_0.stl")
print("Exportada a taza_0.stl")

# Descargarla a tu ordenador (solo en Google Colab; en VS Code/local se omite)
try:
    from google.colab import files
    files.download("taza_0.stl")
except Exception:
    print("(Fuera de Colab: el archivo taza_0.stl esta en tu carpeta de trabajo)")

## 8. (Opcional) Guardar las tazas en tu Google Drive
Así no se borran al cerrar Colab. Descomenta y ejecuta.

In [ ]:
# from google.colab import drive
# import shutil, os
# drive.mount('/content/drive')
# destino = '/content/drive/MyDrive/TFM_tazas/'
# os.makedirs(destino, exist_ok=True)
# for uid, ruta in objetos.items():
#     shutil.copy(ruta, destino)
# print('Tazas copiadas a', destino)

## ¿Y ahora qué?

- Ya tienes tazas 3D descargadas y sabes cargarlas, verlas y exportarlas. ¡Ese es el manejo básico de datos del proyecto!
- **Siguiente paso natural:** generar imágenes de cada taza desde varios ángulos (etapa 1) renderizando con Blender, o 'romper' una taza para preparar datos de reparación (etapa 3, Parte 2 del *Tutorial práctico*).
- Recuerda: para que persistan entre sesiones, usa la celda 8 (Google Drive).